# Khibrah: job-posting pre-processing, step by step

Each pre-processing step runs in its own cell, with the number of rows before and after and examples of what changed.
The same steps run in one call with `pipe.run_file(...)`, which is what the forecasting models and the scraper will use later.

**Before running:** upload `khibrah-jobprep.zip` and `jobarabi_combined.csv` to this Colab session (folder icon on the left), then *Runtime → Run all*.

In [ ]:
# Setup (Colab). Skip the first two lines when running locally from the unzipped folder.
!unzip -oq khibrah-jobprep.zip -d /content
!pip install -q -e /content/khibrah-jobprep
import sys; sys.path.insert(0, '/content/khibrah-jobprep')

import pandas as pd, matplotlib.pyplot as plt
from jobprep import JobPostingPipeline, PipelineConfig, read_any, auto_adapter
from jobprep import plots
pd.set_option('display.max_colwidth', 140)

DATA = 'jobarabi_combined.csv'   # or a Google Drive path after drive.mount('/content/drive')
cfg = PipelineConfig()           # every threshold below can be changed here
pipe = JobPostingPipeline(cfg)

## Load the raw data

In [ ]:
raw = read_any(DATA)
print(raw.shape)
raw.head(3)

## Step 0: Drop empty columns (null handling, columns)
Columns that are at least 95% missing carry no information and are dropped.

In [ ]:
profile = pipe.profile_nulls(raw)
plots.plot_null_profile(profile); plt.show()
raw0 = pipe.drop_null_columns(raw)
print('dropped:', profile.loc[profile.action == 'dropped', 'column'].tolist())
print(raw.shape, '->', raw0.shape)

## Step 1: Map to the common schema
The adapter converts this source's columns into the canonical columns every later step uses. A scraped job board uses a different adapter and the same later steps.

In [ ]:
adapter = auto_adapter(raw0)
df = pipe.adapt(raw0, adapter)
print('adapter:', adapter.name, '| rows:', len(df))
df[['source', 'source_record_id', 'content_type', 'author', 'country_meta', 'posted_at_raw', 'body_raw']].head()

## Step 2: Remove duplicate records
The same tweet captured twice (same record id or same URL) is removed. Reposts of the same advertisement by different accounts are handled later in Step 12.

In [ ]:
df = pipe.remove_duplicate_records(df)
print('removed:', (df.invalid_reason == 'duplicate_record').sum())
df.loc[df.invalid_reason == 'duplicate_record', ['source_url', 'body_raw']].head()

## Step 3: Parse dates
Timestamps become UTC, then the Riyadh local date and the month.

In [ ]:
df = pipe.parse_dates(df)
df[['posted_at_raw', 'posted_at_utc', 'posted_date_local', 'posted_month']].head()

## Step 4: Remove missing values (null handling, rows)
A row with no text or no usable posting date cannot be used for forecasting, so it is removed. Optional fields (salary, city, ...) are left empty and never imputed.

In [ ]:
before = df[['title_raw', 'body_raw', 'posted_at_utc']].isna().sum()
df = pipe.remove_missing(df)
print('missing before removal:'); print(before)
print('\nremoved:', df.invalid_reason.isin(['missing_text', 'missing_date']).sum())

## Step 5: Clean the text
URLs, @mentions, emails, emojis, HTML, tatweel and invisible characters are removed; hashtags become words; Arabic-Indic digits become 0-9.
Spelling is not changed here, so `text_clean` is what the LLM and the virtual assistant read. Removed items are kept in their own columns.

In [ ]:
df = pipe.clean(df)
v = df[df.is_valid]
v[['body_raw', 'text_clean', 'urls', 'hashtags', 'mentions', 'emoji_count']].sample(6, random_state=1)

In [ ]:
plots.plot_before_after(v.body_raw.str.len().mean(), v.text_clean.str.len().mean(),
                        'Average characters per post', ('raw text', 'after cleaning')); plt.show()

## Step 6: Remove invalid text
Placeholders such as *This account doesn't exist* and texts shorter than 15 characters after cleaning are removed.

In [ ]:
df = pipe.remove_invalid_text(df)
df.loc[df.invalid_reason.isin(['placeholder_text', 'too_short']), ['invalid_reason', 'body_raw', 'text_clean']].head(8)

## Step 7: Detect language
Script-based: `ar`, `en`, `fr`, `mixed` (substantial Arabic and Latin), `latin` (other Latin-script) or `unknown`.

In [ ]:
df = pipe.detect_language(df)
v = df[df.is_valid]
plots.plot_categories(v.lang, 'Language of posts', 'Script-based detection on the cleaned text'); plt.show()
v.loc[v.lang == 'mixed', ['text_clean', 'arabic_ratio', 'lang_primary', 'lang_secondary']].head()

## Step 8: Normalize
`text_norm` folds spelling variants so that matching, keyword rules, TF-IDF and de-duplication treat them as the same word:
diacritics removed, أ/إ/آ → ا, ى → ي, ة → ه, Persian letters folded, Latin lower-cased and de-accented, punctuation removed (C++ and C# kept).

In [ ]:
df = pipe.normalize(df)
df.loc[df.is_valid, ['text_clean', 'text_norm']].sample(6, random_state=3)

## Step 9: Find locations
Saudi and foreign cities and countries named in each post, used by Step 11 to keep Saudi Arabia only.

In [ ]:
df = pipe.find_locations(df)
df.loc[df.is_valid, ['text_clean', 'cities_mentioned', 'countries_mentioned']].sample(8, random_state=2)

## Step 10: Flag non-job posts
Job seekers, scam warnings, course advertisements and labour-market news are flagged `is_job_posting = False` (kept in the audit file, excluded from the clean file).

In [ ]:
df = pipe.filter_relevance(df)
v = df[df.is_valid]
print('flagged as not a job ad:', (~v.is_job_posting).sum())
v.loc[~v.is_job_posting, ['text_clean', 'job_reason']].head(10)

## Step 11: Keep Saudi Arabia only
Each post is assigned a country from the strongest evidence available:
1. a Saudi or foreign city/country named in the text (`text_target` / `text_other_country`);
2. otherwise the source's location metadata, which for X/Twitter is the author's profile country (`metadata_target` / `metadata_other_country`);
3. otherwise `unknown`, excluded by default.

Posts outside Saudi Arabia are flagged `in_scope = False`. Set `cfg.scope.target_country = None` to keep every country.

In [ ]:
df = pipe.filter_scope(df)
v = df[df.is_valid]
plots.plot_scope(v); plt.show()
plots.plot_categories(v.job_country.fillna('unknown'), 'Country of each post', 'Top 12 countries',
                      highlight={'Saudi Arabia'}); plt.show()
print('in scope (Saudi Arabia):', v.in_scope.sum(), '| outside:', (~v.in_scope).sum())

## Step 12: Remove duplicate advertisements
Aggregator accounts repost the same advertisement many times. Copies are grouped by exact match and by near-duplicate similarity (MinHash, Jaccard ≥ 0.80 on `text_norm`).
Copies posted within 30 days of the first one count once; a repost after 30 days counts as new demand. The earliest copy is kept.

In [ ]:
df = pipe.deduplicate(df)
v = df[df.is_valid]
plots.plot_duplicate_groups(v.assign(dup_group_size=v.dup_group_size.astype(int))); plt.show()
biggest = v.groupby('dup_group_id').size().idxmax()
v.loc[v.dup_group_id == biggest, ['posted_at_utc', 'author', 'is_duplicate', 'text_clean']].head(5)

## Result

In [ ]:
result = pipe.finish(df, rows_in=len(raw), adapter_name=adapter.name)
print(len(result.postings), 'kept rows |', len(result.rejected), 'removed rows |', len(result.clean), 'unique Saudi job ads in the clean file')
result.step_log[['step', 'rows_before', 'rows_after', 'removed', 'flagged', 'action']]

In [ ]:
plots.plot_step_funnel(result.step_log); plt.show()
plots.plot_monthly(result.clean); plt.show()
from jobprep.lexicons import CITIES
saudi_cities = [c for c, (country, _) in CITIES.items() if country == 'Saudi Arabia']
cities = result.clean.cities_mentioned.explode().dropna()
plots.plot_categories(cities[cities.isin(saudi_cities)], 'Saudi cities named in unique job ads',
                      'A post naming several cities counts once per city'); plt.show()

## Cleaned data
The file for skill extraction: the content, date, source and place of each unique Saudi job advertisement. `text_clean` is for the LLM; `text_norm` is for matching skills against the ontology.

In [ ]:
clean = result.clean
print(f"{len(clean):,} rows × {clean.shape[1]} columns")
clean.head(10)

## Save

In [ ]:
# The cleaned, model-ready data (utf-8-sig keeps Arabic readable in Excel)
clean.to_csv('jobarabi_clean_saudi.csv', index=False, encoding='utf-8-sig')
print('saved', len(clean), 'rows')

# All outputs: postings_clean, postings_all (audit, every column), rejected and report.json
paths = result.save('processed', formats=('csv', 'parquet'))
print(paths)

try:                                   # download in Colab; ignored when running locally
    from google.colab import files
    files.download('jobarabi_clean_saudi.csv')
except ImportError:
    pass

## The same pipeline in one call
This is how the models and the scraper use it. It gives identical results to the steps above.

In [ ]:
same = JobPostingPipeline(cfg).run_file(DATA)
print(len(same.clean) == len(result.clean))

# Scraped job-board records (column names are matched automatically)
sample = [
    {'job_id': 'b1', 'title': 'Data Analyst', 'company': 'Acme', 'location': 'Riyadh',
     'description': '<p>SQL, Power BI, 2+ years of experience. Hybrid.</p>', 'date_posted': '2 days ago', 'scraped_at': '2026-10-24'},
    {'job_id': 'b2', 'title': 'أخصائي موارد بشرية', 'company': 'شركة الأفق', 'location': 'جدة',
     'description': 'بكالوريوس موارد بشرية، خبرة 3 سنوات، للسعوديين فقط، الراتب 9,000 ريال', 'date_posted': '2026-10-20'},
]
JobPostingPipeline(cfg).run_records(sample, source='demo_board').clean